[![cloudevel](img/cloudevel.png)](https://cloudevel.com)

# Sesiones remotas y transmisión de datos.

## Sesiones en servidores.

Desde un inicio UNIX fue diseñado para dar servicio a múltiples usuarios de forma concurrente y generalmente de manera remota.

Para que un usuario accceda a un servidor UNIX o GNU/Linux de forma remota requiere de una conexión.

## Herramientas de conexión.

Las conexiones iniciales se realizaban mediante servicios y protocolos muy básicos e inseguros.

Los servicios más comunes eran:

* *telnet*, el cual es capaz de realiza conexiones por medio de puertos seriales y por medio de redes TCP/IP.

* *ftp*. FTP son las siglas en inglés de Protocolo de Transmisión de archivos y fue diseñado expresamente para ese fin.

    Esl problema con estos servicios y protocolos es que la transmisión de datos es insegura.

## El servicio Secure Shell (*SSH*).

Este es un servicio basado en la arquitectura cliente-servidor capaz de realizar conexiones con las siguientes características:

* Utiliza sistemas de autenticación avanzada.
* Las comunicaciónes son cifradas.
* Permite el manejo de diversos tipos de estándares de cifrado y autenticación.
* Es capaz de crear "túneles" para acceder a recursos de la red local del servidor.

### El servidor `sshd`.

El demonio `sshd` escucha por defecto en el puerto **22** y permite el acceso de los clientes con diversas formas de autenticación. En *Ubuntu* 24.04 la unidad de *systemd* se llama `ssh.service`; también puede estar presente `ssh.socket` (activación por socket). No se debe permitir el inicio de sesión directo de *root* (`PermitRootLogin no`), en línea con las recomendaciones del cuaderno 28.

La configuración adicional se coloca en `/etc/ssh/sshd_config.d/`. Tras editarla, valídela con `sudo sshd -t` y recargue el servicio con `sudo systemctl reload ssh`.


La siguiente celda ejecuta un comando de `apt`.


In [ ]:
sudo apt install openssh-server -y

La siguiente celda inicia el servicio `ssh` (y la que sigue consulta `ssh.service` y `ssh.socket`).

Este ejercicio requiere una máquina Ubuntu 24.04 con *systemd* como proceso de identificador 1 (*PID 1*). No corre en *GitHub Codespaces*.

**Salida en GitHub Codespaces (verificada):**

```
"systemd" is not running in this container due to its overhead.
Use the "service" command to start services instead. e.g.:

service --status-all
```

**Salida en una máquina Ubuntu 24.04 (salida típica):**

```
● ssh.service - OpenBSD Secure Shell server
     Loaded: loaded (/usr/lib/systemd/system/ssh.service; enabled)
     Active: active (running)
```


In [ ]:
sudo systemctl start ssh


La siguiente celda consulta el estado de un servicio con `systemctl`.


In [ ]:
systemctl status ssh --no-pager
systemctl status ssh.socket --no-pager


### Conexión de cliente `ssh`.

Actualmente *SSH* es la forma más común de realizar conexiones remotas a servidores en Internet.

```
ssh <usuario>@<servidor> -p <puerto>
```

El puerto por defecto es el 22 y puede omitirse `-p 22`. Las rutas de archivos no forman parte de la sintaxis de `ssh`: para copiar archivos use `scp` o `rsync`, que sí aceptan `<usuario>@<servidor>:<ruta>`.


**Ejemplo:**

```
ssh usuario@servidor.example.com
```


**Nota:** En Windows 10 ya se puede hablitar un cliente *SSH* nativo. Sin embargo, el cliente más común de SSH es [*Putty*](https://www.putty.org/).

### Generación de una llave *SSH*.

```
ssh-keygen -t ed25519 -C "comentario identificador"
```

Se recomienda el algoritmo *Ed25519*. El comando crea `~/.ssh/id_ed25519` (llave privada) e `id_ed25519.pub` (llave pública). Proteja la llave privada con una frase de paso y no la copie fuera de su equipo.

Si se invoca `ssh-keygen` sin argumentos, algunas versiones aún proponen *RSA* (`id_rsa`); indíque `-t ed25519` de forma explícita.


### El archivo `~/.ssh/known_hosts`.

En la primera conexión a un servidor, el cliente *SSH* guarda la huella de la llave del anfitrión en `~/.ssh/known_hosts`. En las conexiones siguientes verifica esa huella para detectar un posible ataque de intermediario (*man-in-the-middle*).

No deshabilite `StrictHostKeyChecking`. Si la huella cambia de forma inesperada, investigue la causa antes de continuar. Para eliminar una entrada obsoleta de forma controlada use `ssh-keygen -R servidor.example.com`.


La siguiente celda muestra la *man page* de `ssh-keygen`.


In [ ]:
man ssh-keygen

### Transferencia de una llave pública.

La siguiente celda muestra la *man page* de `ssh-copy-id`.


In [ ]:
man ssh-copy-id

### El archivo ```~/.ssh/config```.

El archivo `~/.ssh/config` permite definir alias y parámetros de conexión para hosts *SSH*, evitando tener que escribir opciones largas en cada comando.

Estructura básica:

```
Host alias
    HostName     dirección-o-ip
    User         usuario
    Port         22
    IdentityFile ~/.ssh/id_ed25519
```

Con esta configuración, `ssh alias` equivale a `ssh -i ~/.ssh/id_ed25519 -p 22 usuario@dirección`.

**Ejemplo de archivo `~/.ssh/config`:**

```
Host servidor-prod
    HostName     192.168.1.10
    User         admin
    IdentityFile ~/.ssh/id_ed25519

Host bastion
    HostName     bastion.ejemplo.com
    User         ops
    Port         2222
```

La siguiente celda ejecuta `mkdir -p ~/.ssh && chmod 700 ~/.ssh`.


In [ ]:
# Crear el directorio .ssh con los permisos correctos si no existe
mkdir -p ~/.ssh && chmod 700 ~/.ssh

La siguiente celda muestra el contenido de `~/.ssh/config`.


In [ ]:
# Ver la configuración SSH actual del usuario
cat ~/.ssh/config 2>/dev/null || echo "(archivo no existe aún)"

### Túneles *SSH*.

Los túneles *SSH* permiten enrutar tráfico de red de forma segura a través de una conexión *SSH*. Son fundamentales en entornos *DevOps* para acceder a servicios internos.

#### Túnel local (```-L```).

Redirige un puerto local hacia un destino a través del servidor *SSH*:

```bash
ssh -L puerto_local:host_destino:puerto_destino usuario@servidor
```

**Ejemplo:** acceder a una base de datos interna (puerto 5432) como si fuera local:

```bash
ssh -L 5432:bd-interna:5432 usuario@bastion
# Ahora: psql -h localhost -p 5432
```

#### Túnel remoto (```-R```).

Expone un puerto local en el servidor *SSH* remoto:

```bash
ssh -R puerto_remoto:localhost:puerto_local usuario@servidor
```

**Ejemplo:** exponer un servidor de desarrollo local en el servidor remoto:

```bash
ssh -R 8080:localhost:3000 usuario@servidor
```

#### Proxy *SOCKS* (*Socket Secure*) dinámico (```-D```).

Crea un proxy *SOCKS5* local que enruta todo el tráfico a través del servidor *SSH*:

```bash
ssh -D 1080 usuario@servidor
```

#### Opciones útiles para túneles.

| Opción | Descripción |
|--------|-------------|
| `-N`   | No ejecutar comandos remotos (solo el túnel) |
| `-f`   | Enviar el proceso al fondo (*background*) |
| `-C`   | Comprimir el tráfico |

La siguiente celda muestra la *man page* de `ssh`.


In [ ]:
# Mostrar las opciones de tunneling en la man page
man ssh | grep -A3 "\-L "

### La man page de ```ssh```.

La siguiente celda muestra la *man page* de `ssh`.


In [ ]:
man ssh

## Transferencias de archivos.

###  ```sftp```.
*sftp*. Esta es una implementación de ftp utilizando ssh.

La siguiente celda muestra la *man page* de `sftp`.


In [ ]:
man sftp

### ```scp```.
*scp*. Que es implementación de  ```cp``` por medio de ssh.

```
scp <archivos de origen> <ruta de destino>
```

La ruta en  el servidor remoto se compone de lo siguiente:

```
<usuario>@<servidor>:<ruta local> 
```

La siguiente celda muestra la *man page* de `scp`.


In [ ]:
man scp

**Ejemplo:**

```
scp ~/archivo.txt usuario@servidor.example.com:~/archivo1.txt
```

El anfitrión `servidor.example.com` es un nombre de documentación; no apunta a un laboratorio real. La ruta va después de los dos puntos, en la sintaxis de `scp` (y de `rsync`), no en la de `ssh`.


###  ```rsync```.

Esta es una herramienta avanzada de sincronización de archivos que permite comparar los archivos existentes y actualizarlos con diversas opciones.

Esta herramientas funciona con conexiones remotas y en sistemas de archivos locales.

La siguiente celda muestra la *man page* de `rsync`.


In [ ]:
man rsync

### ```wget ```.

Esta herramienta utiliza el método *GET* de *HTTP* para acceder a recursos publicados mediante su URL.

La siguiente celda ejecuta `wget -O /tmp/cd101-ejemplo.html https://example.com/`.


In [ ]:
wget -O /tmp/cd101-ejemplo.html https://example.com/


### ```curl```.

Curl es una implementación de la biblioteca [lirbcurl](https://curl.haxx.se/libcurl/), la cual permite acceder a recursos publ;icados en Internet por medio de múltiples protocolos.

La siguiente celda muestra la *man page* de `curl`.


In [ ]:
man curl 

### ```git```

[Git](https://git-scm.com/) es un gestor de versiones capaz de sincroniza repositorios locales y remotos.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>